# 第 6 章 · Planning（CrewAI）

书里的做法：把"先列计划、再按计划写"写进**一条 task 的两句指令**里，
由同一个 agent 在同一个回合内完成。

⚠️ 注意 `process=Process.sequential` 只决定"任务之间怎么排"，**它不做规划**——
规划藏在 `Task.description` 那两句话里。真正的规划形态（计划成为可见的中间产物）
见 `../../ch06_planning/README.md`。

kernel 选 **`Python (agentic-crewai)`**。

In [1]:
import os
import threading

from crewai import Agent, Task, Crew, Process, LLM

# 【适配】CrewAI 1.15 改用原生 provider：Agent.llm 只接受 str | BaseLLM，不再收 LangChain 对象，
#   且 "openai/xxx" 这种前缀要走 litellm 回落（未安装）。deepseek 是内置 provider，用 deepseek/ 前缀。
llm = LLM(model="deepseek/deepseek-flash",
          api_key=os.environ["OPENAI_API_KEY"],
          base_url="https://api.deepseek.com/v1")

# ★ 本章新增：规划被写成 task 里的两步指令 —— 先列 bullet-point 计划，再按计划写。
#   计划不落盘、不跨回合、不经过审阅，本质是"让模型先打个草稿再动笔"。
planner_writer_agent = Agent(
    role='Article Planner and Writer',
    goal='Plan and then write a concise, engaging summary on a specified topic.',
    backstory=(
        'You are an expert technical writer and content strategist. '
        'Your strength lies in creating a clear, actionable plan before writing, '
        'ensuring the final summary is both informative and easy to digest.'
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

topic = "The importance of Reinforcement Learning in AI"
high_level_task = Task(
    description=(
        f"1. Create a bullet-point plan for a summary on the topic: '{topic}'.\n"
        f"2. Write the summary based on your plan, keeping it around 200 words."
    ),
    expected_output=(
        "A final report containing two distinct sections:\n\n"
        "### Plan\n"
        "- A bulleted list outlining the main points of the summary.\n\n"
        "### Summary\n"
        "- A concise and well-structured summary of the topic."
    ),
    agent=planner_writer_agent,
)

# ★ 本章新增 Process：决定"任务之间怎么排"。sequential = 挨个跑，不是并行、也不是规划。
crew = Crew(
    agents=[planner_writer_agent],
    tasks=[high_level_task],
    process=Process.sequential,
)


# 【适配】kernel 里有运行中的事件循环，同步 kickoff() 会抛 RuntimeError → 放进独立线程（同第 5 章）
def run_in_thread(fn):
    box = {}

    def _target():
        try:
            box["result"] = fn()
        except BaseException as e:               # noqa: BLE001
            box["error"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


print("## Running the planning and writing task ##")
result = run_in_thread(crew.kickoff)

print("\n\n---\n## Task Result ##\n---")
print(result)

## Running the planning and writing task ##


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Article Planner and Writer                                                                              │
│                                                                                                                 │
│  Task: 1. Create a bullet-point plan for a summary on the topic: 'The importance of Reinforcement Learning in   │
│  AI'.                                                                                                           │
│  2. Write the summary based on your plan, keeping it around 200 words.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Article Planner and Writer                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Plan                                                                                                       │
│  - Define Reinforcement Learning (RL) as learning through interaction, rewards, and trial-and-error.            │
│  - Explain why RL matters: it enables goal-directed decision-making beyond static prediction.                   │
│  - Highlight key strengths: sequential decision-making, adaptivity, and long-term optimization.                 │
│  - Note major breakthroughs and applications: games, robotics, autonomous systems, recommendations, and LLM     │
│  alignment.                                                                                                     │
│  - Mention challenges: sample inefficiency, reward design, stability, and real-world safety.                    │
│  - Conclude that RL is foundational for autonomous, adaptive AI systems.                                        │
│                                                                                                                 │
│  ### Summary                                                                                                    │
│  Reinforcement learning (RL) is important in AI because it shifts the focus from pattern recognition to         │
│  goal-directed decision-making. Instead of relying only on labeled data, RL agents learn by interacting with    │
│  an environment, receiving rewards, and adjusting behavior to maximize long-term outcomes. This makes RL        │
│  essential for problems involving sequential choices, where today’s action affects tomorrow’s options—from      │
│  robotics and autonomous vehicles to resource management and personalized recommendations. RL has also driven   │
│  breakthrough achievements, including superhuman play in games and advances in optimizing complex systems.      │
│  More recently, it underpins alignment and reasoning work in large language models, helping AI systems improve  │
│  through feedback rather than static supervision. Its importance also lies in enabling adaptivity: RL agents    │
│  can improve with experience and respond to changing conditions. However, RL faces challenges such as sample    │
│  inefficiency, unstable training, reward design, and safety in real-world deployment. Addressing these issues   │
│  is critical as AI moves toward autonomous agents that must act reliably in open environments. Overall, RL is   │
│  a foundational technology for AI that can not only predict but decide, adapt, and optimize over time.          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



---
## Task Result ##
---
### Plan
- Define Reinforcement Learning (RL) as learning through interaction, rewards, and trial-and-error.
- Explain why RL matters: it enables goal-directed decision-making beyond static prediction.
- Highlight key strengths: sequential decision-making, adaptivity, and long-term optimization.
- Note major breakthroughs and applications: games, robotics, autonomous systems, recommendations, and LLM alignment.
- Mention challenges: sample inefficiency, reward design, stability, and real-world safety.
- Conclude that RL is foundational for autonomous, adaptive AI systems.

### Summary
Reinforcement learning (RL) is important in AI because it shifts the focus from pattern recognition to goal-directed decision-making. Instead of relying only on labeled data, RL agents learn by interacting with an environment, receiving rewards, and adjusting behavior to maximize long-term outcomes. This makes RL essential for problems involving sequential choices, where today’